# Scanner

> High-level FileScanner class with caching and provider support.

In [ ]:
#| default_exp scanning.scanner

In [ ]:
#| export
import asyncio
from typing import Any, Dict, List, Optional

from cjm_file_discovery.core.models import FileInfo, FileType
from cjm_file_discovery.core.config import ScanConfig, ExtensionMapping
from cjm_file_discovery.cache.memory import ScanCache
from cjm_file_discovery.providers.local import LocalDiscoveryProvider
from cjm_file_discovery.scanning.filters import filter_files, sort_files, limit_files
from cjm_file_discovery.utils.formatting import format_file_size

## FileScanner

High-level file scanner with caching, sorting, and summary statistics.

In [ ]:
#| export
class FileScanner:
    """High-level file scanner with caching and provider support."""

    def __init__(
        self,
        config: ScanConfig,  # Scan configuration
        provider: Optional[Any] = None  # Discovery provider (defaults to LocalDiscoveryProvider)
    ):
        """Initialize the scanner."""
        self.config = config
        self.provider = provider or LocalDiscoveryProvider()
        self._cache = ScanCache(duration_seconds=config.cache_duration_seconds)

    def scan(
        self,
        force_refresh: bool = False  # Force fresh scan, ignoring cache
    ) -> List[FileInfo]:  # List of discovered files
        """Scan for files, using cache if valid."""
        # Check cache first
        if not force_refresh and self.config.cache_results:
            cached = self._cache.get()
            if cached is not None:
                return cached

        # Perform scan
        files = self.provider.scan(self.config.directories, self.config)

        # Sort results
        files = sort_files(files, self.config.sort_by, self.config.sort_descending)

        # Apply limit
        files = limit_files(files, self.config.max_results)

        # Update cache
        if self.config.cache_results:
            self._cache.set(files)

        return files

    async def scan_async(
        self,
        force_refresh: bool = False  # Force fresh scan, ignoring cache
    ) -> List[FileInfo]:  # List of discovered files
        """Async scan for files."""
        # Check cache first
        if not force_refresh and self.config.cache_results:
            cached = self._cache.get()
            if cached is not None:
                return cached

        # Perform async scan
        files = await self.provider.scan_async(self.config.directories, self.config)

        # Sort results
        files = sort_files(files, self.config.sort_by, self.config.sort_descending)

        # Apply limit
        files = limit_files(files, self.config.max_results)

        # Update cache
        if self.config.cache_results:
            self._cache.set(files)

        return files

    def get_files_by_type(
        self,
        file_types: List[FileType]  # File types to filter by
    ) -> List[FileInfo]:  # Filtered files
        """Get files filtered by specific file types."""
        files = self.scan()
        return [f for f in files if f.file_type in file_types]

    def clear_cache(self) -> None:
        """Clear the scan cache."""
        self._cache.clear()

    @property
    def cache_valid(self) -> bool:  # True if cache is valid
        """Check if cache is valid."""
        return self._cache.is_valid()

    @property
    def cached_file_count(self) -> int:  # Number of cached files
        """Get number of cached files."""
        return self._cache.file_count

    def get_summary(self) -> Dict[str, Any]:  # Summary statistics
        """Get summary statistics for scanned files."""
        files = self.scan()

        # Total stats
        total_size = sum(f.size for f in files if f.size is not None)

        # By type breakdown
        by_type: Dict[str, int] = {}
        for f in files:
            type_name = f.file_type.value
            by_type[type_name] = by_type.get(type_name, 0) + 1

        # By extension breakdown (top 10)
        by_extension: Dict[str, int] = {}
        for f in files:
            if f.extension:
                by_extension[f.extension] = by_extension.get(f.extension, 0) + 1
        # Sort by count and take top 10
        by_extension = dict(sorted(by_extension.items(), key=lambda x: x[1], reverse=True)[:10])

        return {
            'total_files': len(files),
            'total_size': total_size,
            'total_size_str': format_file_size(total_size),
            'by_type': by_type,
            'by_extension': by_extension,
            'directories': self.config.directories,
            'cache_valid': self.cache_valid
        }

In [ ]:
import tempfile
from pathlib import Path

from cjm_file_discovery.core.config import FilterConfig

# Test FileScanner with a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    # Create test files
    (Path(tmpdir) / "script.py").write_text("print('hello')")
    (Path(tmpdir) / "data.json").write_text('{"key": "value"}')
    (Path(tmpdir) / "readme.txt").write_text("This is a readme")
    (Path(tmpdir) / "song.mp3").write_bytes(b"\x00" * 1024)  # 1KB fake mp3
    
    # Create scanner
    config = ScanConfig(
        directories=[tmpdir],
        recursive=True,
        cache_results=True,
        cache_duration_seconds=60,
        sort_by="name"
    )
    scanner = FileScanner(config)
    
    # Test scan
    files = scanner.scan()
    assert len(files) == 4
    assert files[0].name == "data.json"  # Sorted by name
    
    # Test caching
    assert scanner.cache_valid == True
    assert scanner.cached_file_count == 4
    
    # Second scan should use cache
    files2 = scanner.scan()
    assert len(files2) == 4
    
    # Force refresh
    files3 = scanner.scan(force_refresh=True)
    assert len(files3) == 4
    
    # Test get_files_by_type
    audio_files = scanner.get_files_by_type([FileType.AUDIO])
    assert len(audio_files) == 1
    assert audio_files[0].name == "song.mp3"
    
    code_files = scanner.get_files_by_type([FileType.CODE])
    assert len(code_files) == 1
    assert code_files[0].name == "script.py"
    
    # Test get_summary
    summary = scanner.get_summary()
    assert summary['total_files'] == 4
    assert summary['total_size'] > 0
    assert 'by_type' in summary
    assert 'by_extension' in summary
    assert summary['cache_valid'] == True
    
    # Test clear_cache
    scanner.clear_cache()
    assert scanner.cache_valid == False

print("FileScanner tests passed!")

FileScanner tests passed!


In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()